# Problema Inverso 3D X33Y33Z33 — Estimativa do Fluxo de Calor no Torneamento

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/x33y33z33_usinagem.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## O problema

Numa ferramenta de corte (torneamento), deseja-se estimar o **fluxo de calor na interface cavaco--ferramenta** — inacessível à medição direta — a partir de **temperaturas medidas** por termopares em seis posições. O método **TFBGF** trata a ferramenta como um sólido 3D (X33Y33Z33: convecção nas seis faces) e estima o fluxo por deconvolução, usando a resposta impulsiva analítica de cada posição.

> 📝 **Nota:** este *notebook* é **auto-contido**: as temperaturas $T_1,\dots,T_6$ são **simuladas** pela própria solução direta (mais ruído), como substitutas das medições reais. Para usar dados experimentais, basta substituir o bloco de geração das temperaturas pela leitura do seu arquivo de medições.

## Bibliotecas
- **NumPy** — computação numérica e FFT.
- **Matplotlib** — gráficos.
- **SciPy** (`optimize.brentq`) — autovalores.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

## Parâmetros, geometria e posições dos termopares

Ferramenta de aço rápido; convecção nas seis faces. O fluxo de corte é imposto numa pequena região da face superior (área $a_p\times f$). As seis posições $T_1,\dots,T_6$ são as da tabela do capítulo.

In [ ]:
k, alpha = 24.0, 7.0868e-6           # aço rápido
L, W, Rc = 12.7e-3, 12.7e-3, 101.6e-3
h = 100.0; Tinf = 26.73; dt = 0.3
t = np.arange(0.0, 120.0 + dt/2, dt); N = len(t)
L1, L2, R1, R2 = 8.28e-3, 10.28e-3, 2.55e-3, 3.505e-3   # região do fluxo
M = 10
POS = {1:(9.28e-3, W, 3.03e-3), 2:(2.76e-3, W, 3.35e-3), 3:(0.0, 5.58e-3, 5.14e-3),
       4:(1.40e-3, W, 9.17e-3), 5:(10.11e-3, W, 9.11e-3), 6:(L, 7.64e-3, 9.88e-3)}

def autovalores(n, Ba, Bb):
    f = lambda g: (g*g - Ba*Bb)*np.sin(g) - g*(Ba+Bb)*np.cos(g)
    r, g, passo = [], 1e-6, np.pi/400
    while len(r) < n and g < (n+3)*np.pi:
        g2 = g + passo
        if f(g)*f(g2) < 0: r.append(brentq(f, g, g2))
        g = g2
    return np.array(r)

B, Bz = h*L/k, h*Rc/k
am = autovalores(M, B, B); bn = autovalores(M, B, B); gp = autovalores(M, Bz, Bz)

## Solução direta e resposta impulsiva

O coeficiente espacial $\Phi$ é o produto das três direções; $A$ é o autovalor combinado. A temperatura integra o fluxo por recorrência; a resposta impulsiva $h$ tem a mesma estrutura com fator $e^{-A t}$. Para a deconvolução usa-se o **núcleo discreto exato** $G$, que satisfaz $T=q*G$ (a relação $dt\,h$ é apenas aproximada nas posições de resposta rápida).

In [ ]:
cf = lambda e, Bb, c, D: (e*np.cos(e*c/D)+Bb*np.sin(e*c/D))/((e**2+Bb**2)*(1+Bb/(e**2+Bb**2))+Bb)
Fx = np.sin(am*L2/L)-np.sin(am*L1/L)-(B/am)*np.cos(am*L2/L)+(B/am)*np.cos(am*L1/L)
Fz = np.sin(gp*R2/Rc)-np.sin(gp*R1/Rc)-(Bz/gp)*np.cos(gp*R2/Rc)+(Bz/gp)*np.cos(gp*R1/Rc)
Amnp = (((am[:,None,None]/L)**2+(bn[None,:,None]/W)**2+(gp[None,None,:]/Rc)**2)*alpha).ravel()
decai = np.exp(-Amnp*dt); onem = 1 - decai; nn = np.arange(N)

def _phi(x, y, z):
    Cx = cf(am, B, x, L); Cy = cf(bn, B, y, W); Cz = cf(gp, Bz, z, Rc)
    return ((Cx*Fx)[:,None,None]*(Cy*(bn*np.cos(bn)+B*np.sin(bn)))[None,:,None]*(Cz*Fz)[None,None,:]).ravel()

def temperatura(x, y, z, q):
    coef = _phi(x, y, z)/Amnp; J = np.zeros(len(Amnp)); T = np.zeros(N)
    for d in range(N):
        J = decai*J + q[d]; T[d] = (alpha/k)*(8/W)*(coef @ (onem*J))
    return Tinf + T

def resposta_impulsiva(x, y, z):
    A = _phi(x, y, z); H = np.zeros(N)
    H[1:] = (alpha/k)*(8/W)*(A @ np.exp(-Amnp[:,None]*t[None,1:]))
    return H

def kernel(x, y, z):
    coef = _phi(x, y, z)/Amnp
    return (alpha/k)*(8/W)*((coef*onem) @ np.exp(-Amnp[:,None]*(dt*nn)[None,:]))

## Temperaturas medidas (simuladas)

Impõe-se um pulso de fluxo de corte (janela de 10 a 60 s) e calcula-se a temperatura em cada posição, somando um ruído gaussiano de medição — o que substitui as medições reais.

In [ ]:
Q = 4.0e6
q_true = np.where(t < 10, 0.0, np.where(t < 15, Q*(t-10)/5,
         np.where(t <= 55, Q, np.where(t <= 60, Q*(60-t)/5, 0.0))))
rng = np.random.default_rng(2024)
T_meas = {i: temperatura(*POS[i], q_true) + rng.normal(0, 0.3, N) for i in POS}

cores = ['#D55E00', '#0072B2', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']
tracos = ['-', '--', '-.', ':', (0, (5, 1)), (0, (3, 1, 1, 1))]
fig, ax = plt.subplots()
for i in range(1, 7):
    ax.plot(t, T_meas[i], color=cores[i-1], linestyle=tracos[i-1], linewidth=1.3, label=f'T{i}')
ax.set_xlabel('Tempo (s)'); ax.set_ylabel('Temperatura  (°C)')
ax.set_title('Temperaturas medidas (simuladas) no torneamento')
ax.legend(ncol=2); ax.grid(True); plt.tight_layout(); plt.show()

## Respostas impulsivas

A resposta impulsiva da **posição 1** (junto da fonte) é estritamente decrescente; as das posições **2–6** sobem antes de decair — daí a necessidade de correção na deconvolução.

In [ ]:
H = {i: resposta_impulsiva(*POS[i]) for i in POS}
fig, ax = plt.subplots()
for i in range(1, 7):
    ax.plot(t[1:], H[i][1:], color=cores[i-1], linestyle=tracos[i-1], linewidth=1.4, label=f'h{i}')
ax.set_xlabel('Tempo (s)'); ax.set_ylabel('h(t)  (m²·K/J)')
ax.set_title('Respostas impulsivas das seis posições'); ax.set_xlim(0, 30)
ax.legend(ncol=2); ax.grid(True); plt.tight_layout(); plt.show()

## Estimativa do fluxo pela posição 1

Com a temperatura $T_1$ e o núcleo $G_1$, a deconvolução via FFT (com filtro passa-baixa regularizador) recupera o fluxo de corte. Realimentando esse fluxo na solução direta, a temperatura estimada reproduz a medida.

In [ ]:
NR = 2**20; fr = np.fft.fftfreq(NR, dt)
def deconv(T, G, fc):
    Wf = 1.0/(1.0 + (np.abs(fr)/fc)**8)
    return np.real(np.fft.ifft(np.fft.fft(T - Tinf, NR)/np.fft.fft(G, NR)*Wf))[:N]

G = {i: kernel(*POS[i]) for i in POS}
q1 = deconv(T_meas[1], G[1], 0.06)
tc = N - 20

fig, (axa, axb) = plt.subplots(1, 2, figsize=(10, 4))
axa.plot(t[:tc], q1[:tc]/1e6, color=cores[0], linewidth=1.5)
axa.set_xlabel('Tempo (s)'); axa.set_ylabel('q estimado  (×10⁶ W/m²)')
axa.set_title('Fluxo de calor estimado (q1)'); axa.grid(True)

T1r = temperatura(*POS[1], q1)
axb.plot(t[:tc], T_meas[1][:tc], color='k', linewidth=1.5, label='medida')
axb.plot(t[:tc:6], T1r[:tc:6], color=cores[0], linestyle='none', marker='o',
         markersize=3.6, markerfacecolor='none', label='estimada')
axb.set_xlabel('Tempo (s)'); axb.set_ylabel('T1  (°C)')
axb.set_title('Posição 1: medida × estimada'); axb.legend(); axb.grid(True)
plt.tight_layout(); plt.show()
print(f'Resíduo médio em T1: {np.mean(np.abs(T1r[10:tc]-T_meas[1][10:tc])):.2f} °C')

## Divergência nas posições 2–6 e a correção da resposta impulsiva

Aplicar a resposta impulsiva **não corrigida** de uma posição retardada produz estimativas **divergentes** (oscilações enormes). A correção consiste em tomar a resposta impulsiva **a partir do seu máximo** (tornando-a estritamente decrescente) antes da deconvolução, e recolocar o atraso ao final. Abaixo, a posição 2 sem e com correção.

In [ ]:
def deconv_corrigida(T, G, fc):
    im = int(np.argmax(G))
    Gc = np.zeros(N); Gc[:N-im] = G[im:]
    qc = deconv(T, Gc, fc)
    return np.concatenate([qc[im:], np.zeros(im)])

q2_div = deconv(T_meas[2], G[2], 0.35)          # sem correção -> diverge
q2_cor = deconv_corrigida(T_meas[2], G[2], 0.06) # com correção

fig, (axa, axb) = plt.subplots(1, 2, figsize=(10, 4))
axa.plot(t[:tc], q2_div[:tc]/1e6, color=cores[1], linewidth=1.0)
axa.set_xlabel('Tempo (s)'); axa.set_ylabel('q  (×10⁶ W/m²)')
axa.set_title('Posição 2 — sem correção (diverge)'); axa.grid(True)
axb.plot(t[:tc], q2_cor[:tc]/1e6, color=cores[1], linewidth=1.5)
axb.set_xlabel('Tempo (s)'); axb.set_ylabel('q  (×10⁶ W/m²)')
axb.set_title('Posição 2 — com correção'); axb.grid(True)
plt.tight_layout(); plt.show()

## Temperatura na interface cavaco–ferramenta

Onde não há sensor, estima-se a temperatura pela solução direta alimentada com o fluxo $q_1$ (posição mais próxima da fonte). Mostra-se também o campo de temperatura na face superior no instante de pico.

In [ ]:
T_if = temperatura(9.28e-3, W, 3.03e-3, q1)     # ponto na região de corte
fig, ax = plt.subplots()
ax.plot(t[:tc], T_if[:tc], color=cores[0], linewidth=1.6, label='interface (estimada)')
ax.plot(t[:tc], T_meas[1][:tc], color='k', linestyle='--', linewidth=1.1, label='T1 (medida)')
ax.set_xlabel('Tempo (s)'); ax.set_ylabel('Temperatura  (°C)')
ax.set_title('Temperatura estimada na interface cavaco–ferramenta')
ax.legend(); ax.grid(True); plt.tight_layout(); plt.show()
print(f'Temperatura máxima estimada na interface: {T_if.max():.1f} °C')